![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/rule-based-matcher/RuleBasedMatcher.ipynb)

# **Advanced RuleBasedMatcher in Spark NLP**

`RuleBasedMatcher` is a token-pattern matcher for Spark NLP annotation columns. It lets you write rules over token text, lemmas, POS tags, NER tags, normalized NER entity types, annotation metadata, and other mapped annotation columns, then emits matched spans as `CHUNK` annotations.

Use it when plain regular expressions are too brittle but a trained model would be unnecessary or unavailable: addresses, job requirements, contact fields, financial phrases, and domain terminology are common examples.

## **0. Colab Setup**

In [2]:
# Only run this cell when you are using Spark NLP on Google Colab
!wget http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [ ]:
!pip install -q spark-nlp-display

## **1. Imports and Spark NLP Session**

In [27]:
import json
import os
import re
import shutil
import sys
import tempfile
import time
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from pyspark.ml import Pipeline, PipelineModel
from pyspark.sql import functions as F

import sparknlp
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import (
    LemmatizerModel,
    NerCrfModel,
    PerceptronModel,
    RuleBasedMatcher,
    SentenceDetector,
    Tokenizer,
    WordEmbeddingsModel,
)
from sparknlp.common import ReadAs
from sparknlp_display import NerVisualizer as visualizer

spark = sparknlp.start()

print("Spark NLP version :", sparknlp.version())
print("Apache Spark version:", spark.version)

## **2. Sample Documents**

The examples below are intentionally mixed: addresses, contact details, job requirements, financial phrases, and product/domain terminology. This lets us reuse one upstream annotation pipeline and then write several different rule groups.

In [5]:
sample_texts = [
    (0, "address", "443 8th Street, New York. 21-B Baker Road, London. 1600 Pennsylvania Avenue, Washington."),
    (1, "contact", "Jane Smith, Senior Data Scientist, joined Example Corp in Berlin. Call phone: +1 415 555 0100 or email: jane.smith@example.com."),
    (2, "jobs", "We need 3+ years of Python experience and at least five years working with Java. Experience in Spark or Scala is preferred. Bachelor's degree in computer science required."),
    (3, "finance", "Revenue increased by 18 percent. Operating loss of $2.4 million was reported. The startup was acquired by Example Corp and is headquartered in Berlin."),
    (4, "domain", "The customer data platform supports real-time event streaming and identity resolution for enterprise accounts."),
]

df = spark.createDataFrame(sample_texts, ["doc_id", "use_case", "text"])
df.show(truncate=False)

+------+--------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|doc_id|use_case|text                                                                                                                                                                       |
+------+--------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0     |address |443 8th Street, New York. 21-B Baker Road, London. 1600 Pennsylvania Avenue, Washington.                                                                                   |
|1     |contact |Jane Smith, Senior Data Scientist, joined Example Corp in Berlin. Call phone: +1 415 555 0100 or email: jane.smith@example.com.                                            |
|2     |jobs    |We need 3+ years of Python experi

## **3. Build the Upstream Annotation Pipeline**

`RuleBasedMatcher` does not create tokens, lemmas, POS tags, or NER tags. It consumes those annotations. Here we build a CPU-friendly public Spark NLP pipeline:

* `DocumentAssembler`, `SentenceDetector`, and `Tokenizer` provide document, sentence, and token spans.
* `LemmatizerModel` gives canonical forms such as `increased` -> `increase`.
* `PerceptronModel` gives POS tags.
* `WordEmbeddingsModel` plus `NerCrfModel` gives token-level NER tags without TensorFlow.

The first run downloads public models. Later runs usually read them from the Spark NLP cache.

In [6]:
document = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentence = SentenceDetector() \
    .setInputCols(["document"]) \
    .setOutputCol("sentence")

token = Tokenizer() \
    .setInputCols(["sentence"]) \
    .setOutputCol("token")

lemma = LemmatizerModel.pretrained("lemma_antbnc", "en") \
    .setInputCols(["token"]) \
    .setOutputCol("lemma")

pos = PerceptronModel.pretrained("pos_anc", "en") \
    .setInputCols(["sentence", "token"]) \
    .setOutputCol("pos")

embeddings = WordEmbeddingsModel.pretrained("glove_100d", "en") \
    .setInputCols(["sentence", "token"]) \
    .setOutputCol("embeddings") \
    .setCaseSensitive(False)

ner = NerCrfModel.pretrained() \
    .setInputCols(["sentence", "token", "pos", "embeddings"]) \
    .setOutputCol("ner")

annotation_pipeline = Pipeline(stages=[document, sentence, token, lemma, pos, embeddings, ner])
annotation_model = annotation_pipeline.fit(df)
annotated = annotation_model.transform(df).cache()

annotated.select("doc_id", "use_case", "text").show(truncate=80)

lemma_antbnc download started this may take some time.
Approximate size to download 907.6 KB
[OK!]
pos_anc download started this may take some time.
Approximate size to download 3.9 MB
[OK!]
glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[OK!]
ner_crf download started this may take some time.
Approximate size to download 10.2 MB
[OK!]
+------+--------+--------------------------------------------------------------------------------+
|doc_id|use_case|                                                                            text|
+------+--------+--------------------------------------------------------------------------------+
|     0| address|443 8th Street, New York. 21-B Baker Road, London. 1600 Pennsylvania Avenue, ...|
|     1| contact|Jane Smith, Senior Data Scientist, joined Example Corp in Berlin. Call phone:...|
|     2|    jobs|We need 3+ years of Python experience and at least five years working with Ja...|
|     3| finance|Revenue 

## **4. Inspect Available Token Attributes Before Writing Rules**

A good rule starts by looking at the annotations the upstream pipeline actually produced. The CRF NER model uses IOB-style tags such as `I-LOC`; `RuleBasedMatcher` can use the raw tag (`NER` or `NER_TAG`) or the normalized entity type (`NER_TYPE`, for example `LOC`).

In [7]:
def normalize_ner_type(tag):
    if tag is None:
        return None
    tag = tag.strip()
    if tag == "O" or not tag:
        return tag
    if "-" in tag:
        prefix, entity = tag.split("-", 1)
        if prefix in {"B", "I", "O", "E", "S", "U", "L"}:
            return entity
    return tag


def token_table(result_df, doc_id, limit=80):
    row = result_df.where(F.col("doc_id") == doc_id).select("text", "token", "lemma", "pos", "ner").first()
    records = []
    for i, (tok, lem, pos_ann, ner_ann) in enumerate(zip(row.token, row.lemma, row.pos, row.ner)):
        records.append({
            "i": i,
            "token": tok.result,
            "lemma": lem.result,
            "pos": pos_ann.result,
            "ner_tag": ner_ann.result,
            "ner_type": normalize_ner_type(ner_ann.result),
            "sentence": tok.metadata.get("sentence"),
            "begin": tok.begin,
            "end": tok.end,
        })
    return pd.DataFrame(records).head(limit)


def match_table(result_df, match_col="rule_matches"):
    rows = result_df.select("doc_id", "use_case", "text", F.explode_outer(match_col).alias("match")) \
        .where(F.col("match").isNotNull()) \
        .collect()
    records = []
    for row in rows:
        meta = row.match.metadata
        records.append({
            "doc_id": row.doc_id,
            "use_case": row.use_case,
            "matched_text": row.match.result,
            "rule_id": meta.get("rule"),
            "label": meta.get("entity") or meta.get("label"),
            "sentence": meta.get("sentence"),
            "char_begin": row.match.begin,
            "char_end": row.match.end,
            "sentence_token_begin": meta.get("sentenceTokenBegin"),
            "sentence_token_end": meta.get("sentenceTokenEnd"),
            "document_token_begin": meta.get("documentTokenBegin"),
            "document_token_end": meta.get("documentTokenEnd"),
            "priority": meta.get("priority"),
            "pattern": meta.get("pattern"),
        })
    return pd.DataFrame(records)


def make_matcher(rules, output_col="rule_matches", overlap_strategy="ALL"):
    return RuleBasedMatcher() \
        .setInputCols(["document", "sentence", "token", "lemma", "pos", "ner"]) \
        .setOutputCol(output_col) \
        .setRules(rules) \
        .setAttributeColumns({
            "TEXT": "token",
            "LOWER": "token",
            "TOKEN": "token",
            "LEMMA": "lemma",
            "POS": "pos",
            "NER": "ner",
            "NER_TAG": "ner",
            "NER_TYPE": "ner",
        }) \
        .setOverlapStrategy(overlap_strategy)


def run_rules(rules, output_col="rule_matches", overlap_strategy="ALL"):
    matcher = make_matcher(rules, output_col=output_col, overlap_strategy=overlap_strategy)
    model = matcher.fit(annotated)
    return model.transform(annotated), model

pd.set_option("display.max_colwidth", 120)

In [8]:
display(token_table(annotated, 0, limit=40))

,i,token,lemma,pos,ner_tag,ner_type,sentence,begin,end
0,0,443,443,CD,O,O,0,0,2
1,1,8th,8th,CD,O,O,0,4,6
2,2,Street,Street,NNP,O,O,0,8,13
3,3,",",",",",",O,O,0,14,14
4,4,New,New,NNP,I-LOC,LOC,0,16,18
5,5,York,York,NNP,I-LOC,LOC,0,20,23
6,6,.,.,.,O,O,0,24,24
7,7,21-B,21-B,JJ,O,O,1,26,29
8,8,Baker,Baker,NNP,I-ORG,ORG,1,31,35
9,9,Road,Road,NNP,I-ORG,ORG,1,37,40


The address document shows why token attributes matter. `New York`, `London`, and `Washington` are tagged as locations, while street words are regular tokens with useful POS tags. A regex over raw text would have to rediscover all of this structure.

## **5. First Rule: Match Normalized Location Tokens**

This first rule is intentionally small. It matches any token whose normalized NER type is `LOC`. It uses `NER_TYPE`, not `NER_TAG`, so it does not need to care whether the raw tag was `B-LOC`, `I-LOC`, `U-LOC`, or another supported prefix.

In [9]:
location_rules = [
    {
        "id": "location_token",
        "label": "LOCATION_TOKEN",
        "patterns": [[{"NER_TYPE": "LOC"}]],
    }
]

location_result, _ = run_rules(location_rules)
display(match_table(location_result).head(20))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,0,address,New,location_token,LOCATION_TOKEN,0,16,18,4,4,4,4,0,0
1,0,address,York,location_token,LOCATION_TOKEN,0,20,23,5,5,5,5,0,0
2,0,address,London,location_token,LOCATION_TOKEN,1,43,48,4,4,11,11,0,0
3,0,address,Pennsylvania,location_token,LOCATION_TOKEN,2,56,67,1,1,14,14,0,0
4,0,address,Avenue,location_token,LOCATION_TOKEN,2,69,74,2,2,15,15,0,0
5,0,address,Washington,location_token,LOCATION_TOKEN,2,77,86,4,4,17,17,0,0
6,1,contact,Berlin,location_token,LOCATION_TOKEN,0,58,63,11,11,11,11,0,0
7,3,finance,Berlin,location_token,LOCATION_TOKEN,2,143,148,11,11,25,25,0,0


## **6. Postal Address Extraction**

Addresses have structure but also variation:

* house number: `443`, `21-B`, `1600`
* optional ordinal-like token: `8th`
* street suffix: `Street`, `Road`, `Avenue`
* optional comma
* one or more location tokens

We use multiple patterns under one rule because the examples are not all the same shape.

In [10]:
address_rules = [
    {
        "id": "postal_address",
        "label": "ADDRESS",
        "priority": 50,
        "patterns": [
            [
                {"TEXT": {"REGEX": "^\\d+[A-Za-z-]*$"}},
                {"TEXT": {"REGEX": "^\\d+(?:st|nd|rd|th)$"}, "OP": "?"},
                {"LOWER": {"IN": ["street", "st", "road", "rd", "avenue", "ave", "lane", "ln"]}},
                {"TEXT": ",", "OP": "?"},
                {"NER_TYPE": {"IN": ["LOC", "GPE"]}, "OP": "+"},
            ],
            [
                {"TEXT": {"REGEX": "^\\d+[A-Za-z-]*$"}},
                {"POS": {"IN": ["NN", "NNS", "NNP", "NNPS", "JJ"]}, "OP": "+"},
                {"LOWER": {"IN": ["street", "st", "road", "rd", "avenue", "ave", "lane", "ln"]}},
                {"TEXT": ",", "OP": "?"},
                {"NER_TYPE": {"IN": ["LOC", "GPE"]}, "OP": "+"},
            ],
        ],
    }
]

address_result, _ = run_rules(address_rules)
display(match_table(address_result))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,0,address,"443 8th Street, New York",postal_address,ADDRESS,0,0,23,0,5,0,5,50,0
1,0,address,"8th Street, New York",postal_address,ADDRESS,0,4,23,1,5,1,5,50,0
2,0,address,"21-B Baker Road, London",postal_address,ADDRESS,1,26,48,0,4,7,11,50,1
3,0,address,"1600 Pennsylvania Avenue, Washington",postal_address,ADDRESS,2,51,86,0,4,13,17,50,1


The two address patterns share one rule ID and label. Downstream consumers see consistent `ADDRESS` chunks even though each span was matched by a different pattern shape.

This example keeps the default overlap strategy, `ALL`, so it may show both a full address and a shorter overlapping address-like span. That is useful while authoring rules because it exposes every candidate from every starting position. Within one starting position, the final pattern element is greedy. Later sections show how `PRIORITY_LONGEST` keeps the highest-priority, longest span for production extraction.

## **7. Contact and Identity Information**

The next rules combine NER, lexical labels, punctuation, regexes, and repetitions:

* person name followed by a job title;
* phone label followed by numeric pieces;
* email label followed by an email token;
* organization followed by a location.

In [11]:
display(token_table(annotated, 1, limit=35))

,i,token,lemma,pos,ner_tag,ner_type,sentence,begin,end
0,0,Jane,Jane,NNP,I-PER,PER,0,0,3
1,1,Smith,Smith,NNP,I-PER,PER,0,5,9
2,2,",",",",",",O,O,0,10,10
3,3,Senior,Senior,NNP,I-ORG,ORG,0,12,17
4,4,Data,Data,NNP,I-ORG,ORG,0,19,22
5,5,Scientist,Scientist,NNP,I-ORG,ORG,0,24,32
6,6,",",",",",",O,O,0,33,33
7,7,joined,join,VBD,O,O,0,35,40
8,8,Example,Example,NN,I-ORG,ORG,0,42,48
9,9,Corp,Corp,NNP,I-ORG,ORG,0,50,53


In [12]:
contact_rules = [
    {
        "id": "person_with_title",
        "label": "PERSON_TITLE",
        "priority": 30,
        "patterns": [[
            {"NER_TYPE": "PER", "OP": "+"},
            {"TEXT": ",", "OP": "?"},
            {"POS": {"IN": ["NNP", "NN", "JJ"]}, "OP": "+"},
        ]],
    },
    {
        "id": "phone_label_value",
        "label": "PHONE",
        "priority": 20,
        "patterns": [[
            {"LOWER": "phone"},
            {"TEXT": ":", "OP": "?"},
            {"TEXT": {"REGEX": "^\\+?\\d+$"}, "OP": "{3,4}"},
        ]],
    },
    {
        "id": "email_label_value",
        "label": "EMAIL",
        "priority": 20,
        "patterns": [[
            {"LOWER": "email"},
            {"TEXT": ":", "OP": "?"},
            {"TEXT": {"REGEX": "^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$"}},
        ]],
    },
    {
        "id": "organization_in_location",
        "label": "ORG_LOCATION",
        "priority": 25,
        "patterns": [[
            {"NER_TYPE": "ORG", "OP": "+"},
            {"LOWER": "in"},
            {"NER_TYPE": "LOC", "OP": "+"},
        ]],
    },
]

contact_result, _ = run_rules(contact_rules)
display(match_table(contact_result))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,1,contact,Jane Smith,person_with_title,PERSON_TITLE,0,0,9,0,1,0,1,30,0
1,1,contact,"Jane Smith, Senior Data Scientist",person_with_title,PERSON_TITLE,0,0,32,0,5,0,5,30,0
2,1,contact,"Smith, Senior Data Scientist",person_with_title,PERSON_TITLE,0,5,32,1,5,1,5,30,0
3,1,contact,Example Corp in Berlin,organization_in_location,ORG_LOCATION,0,42,63,8,11,8,11,25,0
4,1,contact,Corp in Berlin,organization_in_location,ORG_LOCATION,0,50,63,9,11,9,11,25,0
5,1,contact,phone: +1 415 555 0100,phone_label_value,PHONE,1,71,92,1,6,14,19,20,0
6,1,contact,email: jane.smith@example.com,email_label_value,EMAIL,1,97,125,8,10,21,23,20,0


The phone and email examples are good fits for regex predicates because the value format is structured. The person/title and organization/location examples benefit from NER and POS tags.

The default `ALL` strategy intentionally keeps overlapping candidates here. Use the overlap controls below when a downstream workflow needs one deterministic span per region of text.

## **8. Lemmas, Alternatives, Optional Tokens, and Wildcards for Job Requirements**

Job requirements vary a lot. Lemmas let us match `working` with `work`, alternatives let one rule cover multiple skills, and bounded wildcards let us skip short connectors without making the rule too broad.

In [13]:
display(token_table(annotated, 2, limit=45))

,i,token,lemma,pos,ner_tag,ner_type,sentence,begin,end
0,0,We,We,PRP,O,O,0,0,1
1,1,need,need,VBP,O,O,0,3,6
2,2,3+,3+,CD,O,O,0,8,9
3,3,years,year,NNS,O,O,0,11,15
4,4,of,of,IN,O,O,0,17,18
5,5,Python,Python,NNP,I-MISC,MISC,0,20,25
6,6,experience,experience,NN,O,O,0,27,36
7,7,and,and,CC,O,O,0,38,40
8,8,at,at,IN,O,O,0,42,43
9,9,least,least,JJS,O,O,0,45,49


In [14]:
job_rules = [
    {
        "id": "years_of_skill_experience",
        "label": "EXPERIENCE_REQUIREMENT",
        "priority": 40,
        "patterns": [[
            {"TEXT": {"REGEX": "^(?:\\d+\\+?|one|two|three|four|five|six|seven|eight|nine|ten)$"}},
            {"LEMMA": "year"},
            {"LOWER": "of", "OP": "?"},
            {"TEXT": {"IN": ["Python", "Java", "Spark", "Scala"]}},
            {"LEMMA": "experience", "OP": "?"},
        ]],
    },
    {
        "id": "experience_in_skill_list",
        "label": "SKILL_REQUIREMENT",
        "priority": 35,
        "patterns": [[
            {"LEMMA": "experience"},
            {"LOWER": "in"},
            {"TEXT": {"IN": ["Python", "Java", "Spark", "Scala"]}},
            {"LOWER": "or", "OP": "?"},
            {"TEXT": {"IN": ["Python", "Java", "Spark", "Scala"]}, "OP": "?"},
        ]],
    },
    {
        "id": "degree_requirement",
        "label": "EDUCATION_REQUIREMENT",
        "priority": 30,
        "patterns": [[
            {"LOWER": {"REGEX": "^bachelor"}},
            {"LEMMA": "degree"},
            {"LOWER": "in"},
            {},
            {"LOWER": "science"},
        ]],
    },
]

job_result, _ = run_rules(job_rules)
display(match_table(job_result))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,2,jobs,3+ years of Python experience,years_of_skill_experience,EXPERIENCE_REQUIREMENT,0,8,36,2,6,2,6,40,0
1,2,jobs,Bachelor's degree in computer science,degree_requirement,EDUCATION_REQUIREMENT,2,124,160,0,4,24,28,30,0


The empty token pattern `{}` is the canonical wildcard. In the degree rule it stands for exactly one token (`computer`). Use wildcard repetitions carefully; broad patterns such as `{"OP": "*"}` are powerful but can be expensive and may match more than intended.

## **9. Financial and Business Phrase Extraction**

Here we combine lexical patterns, lemmas, POS tags, NER, regex values, and token metadata. The `META.sentence` predicate below reads token metadata from Spark NLP annotations; it is useful when a custom upstream annotator writes domain metadata onto tokens.

In [15]:
display(token_table(annotated, 3, limit=45))

,i,token,lemma,pos,ner_tag,ner_type,sentence,begin,end
0,0,Revenue,Revenue,NNP,O,O,0,0,6
1,1,increased,increase,VBN,O,O,0,8,16
2,2,by,by,IN,O,O,0,18,19
3,3,18,18,CD,O,O,0,21,22
4,4,percent,percent,NN,O,O,0,24,30
5,5,.,.,.,O,O,0,31,31
6,6,Operating,Operating,NNP,O,O,1,33,41
7,7,loss,loss,NN,O,O,1,43,46
8,8,of,of,IN,O,O,1,48,49
9,9,$2.4,$2.4,NN,O,O,1,51,54


In [16]:
finance_rules = [
    {
        "id": "revenue_change_first_sentence",
        "label": "REVENUE_CHANGE",
        "priority": 45,
        "patterns": [[
            {"LOWER": "revenue", "META.sentence": "0"},
            {"LEMMA": "increase"},
            {"LOWER": "by"},
            {"POS": "CD"},
            {"LOWER": {"IN": ["percent", "%"]}},
        ]],
    },
    {
        "id": "operating_loss_amount",
        "label": "LOSS_AMOUNT",
        "priority": 40,
        "patterns": [[
            {"LOWER": "operating"},
            {"LEMMA": "loss"},
            {"LOWER": "of"},
            {"TEXT": {"REGEX": "^[$€£]?\\d+(?:\\.\\d+)?$"}},
            {"LOWER": {"IN": ["million", "billion"]}},
        ]],
    },
    {
        "id": "acquired_by_org",
        "label": "ACQUISITION",
        "priority": 35,
        "patterns": [[
            {"LEMMA": "acquire"},
            {"LOWER": "by"},
            {"NER_TYPE": "ORG", "OP": "+"},
        ]],
    },
    {
        "id": "headquartered_in_location",
        "label": "HEADQUARTERS",
        "priority": 35,
        "patterns": [[
            {"LEMMA": "headquarter"},
            {"LOWER": "in"},
            {"NER_TYPE": "LOC", "OP": "+"},
        ]],
    },
]

finance_result, _ = run_rules(finance_rules)
display(match_table(finance_result))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,3,finance,Revenue increased by 18 percent,revenue_change_first_sentence,REVENUE_CHANGE,0,0,30,0,4,0,4,45,0
1,3,finance,Operating loss of $2.4 million,operating_loss_amount,LOSS_AMOUNT,1,33,62,0,4,6,10,40,0
2,3,finance,acquired by Example Corp,acquired_by_org,ACQUISITION,2,94,117,3,6,17,20,35,0
3,3,finance,headquartered in Berlin,headquartered_in_location,HEADQUARTERS,2,126,148,9,11,23,25,35,0


## **10. Raw NER Tags vs Normalized NER Types**

Use `NER_TAG` or `NER` when the prefix matters. Use `NER_TYPE` when you want the entity type regardless of BIO/BILOU-style prefix.

In [17]:
ner_comparison_rules = [
    {"id": "raw_i_loc", "label": "RAW_I_LOC", "patterns": [[{"NER_TAG": "I-LOC"}]]},
    {"id": "normalized_loc", "label": "NORMALIZED_LOC", "patterns": [[{"NER_TYPE": "LOC"}]]},
]

ner_comparison, _ = run_rules(ner_comparison_rules)
display(match_table(ner_comparison).head(20))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,0,address,New,raw_i_loc,RAW_I_LOC,0,16,18,4,4,4,4,0,0
1,0,address,New,normalized_loc,NORMALIZED_LOC,0,16,18,4,4,4,4,0,0
2,0,address,York,raw_i_loc,RAW_I_LOC,0,20,23,5,5,5,5,0,0
3,0,address,York,normalized_loc,NORMALIZED_LOC,0,20,23,5,5,5,5,0,0
4,0,address,London,raw_i_loc,RAW_I_LOC,1,43,48,4,4,11,11,0,0
5,0,address,London,normalized_loc,NORMALIZED_LOC,1,43,48,4,4,11,11,0,0
6,0,address,Pennsylvania,raw_i_loc,RAW_I_LOC,2,56,67,1,1,14,14,0,0
7,0,address,Pennsylvania,normalized_loc,NORMALIZED_LOC,2,56,67,1,1,14,14,0,0
8,0,address,Avenue,raw_i_loc,RAW_I_LOC,2,69,74,2,2,15,15,0,0
9,0,address,Avenue,normalized_loc,NORMALIZED_LOC,2,69,74,2,2,15,15,0,0


Both rules match the same locations in this CRF model because it emits `I-LOC`. If a different model emits `B-LOC`, `U-LOC`, or `S-LOC`, the `NER_TYPE` rule still matches `LOC`.

## **11. Overlapping and Competing Matches**

Overlaps are common when one rule is a short version of another. The default `ALL` strategy returns every candidate from every starting position; the final pattern element remains greedy within one start. `PRIORITY_LONGEST` prefers higher priority and then longer spans.

In [18]:
overlap_rules = [
    {
        "id": "python_skill_short",
        "label": "SKILL",
        "priority": 10,
        "patterns": [[{"TEXT": "Python"}, {"LEMMA": "experience"}]],
    },
    {
        "id": "python_experience_long",
        "label": "EXPERIENCE_REQUIREMENT",
        "priority": 50,
        "patterns": [[
            {"TEXT": {"REGEX": "^\\d+\\+?$"}},
            {"LEMMA": "year"},
            {"LOWER": "of"},
            {"TEXT": "Python"},
            {"LEMMA": "experience"},
        ]],
    },
]

overlap_all, _ = run_rules(overlap_rules, output_col="overlap_all", overlap_strategy="ALL")
overlap_priority, _ = run_rules(overlap_rules, output_col="overlap_priority", overlap_strategy="PRIORITY_LONGEST")

print("ALL")
display(match_table(overlap_all, "overlap_all"))
print("PRIORITY_LONGEST")
display(match_table(overlap_priority, "overlap_priority"))

ALL


,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,2,jobs,3+ years of Python experience,python_experience_long,EXPERIENCE_REQUIREMENT,0,8,36,2,6,2,6,50,0
1,2,jobs,Python experience,python_skill_short,SKILL,0,20,36,5,6,5,6,10,0


PRIORITY_LONGEST


,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,2,jobs,3+ years of Python experience,python_experience_long,EXPERIENCE_REQUIREMENT,0,8,36,2,6,2,6,50,0


## **12. Domain-Specific Terminology Without a Trained Model**

Domain terminology often has stable heads but variable modifiers. These rules detect phrases around data platforms, event streaming, identity resolution, and enterprise accounts without training a classifier or NER model.

In [19]:
display(token_table(annotated, 4, limit=30))

,i,token,lemma,pos,ner_tag,ner_type,sentence,begin,end
0,0,The,The,DT,O,O,0,0,2
1,1,customer,customer,NN,O,O,0,4,11
2,2,data,data,NNS,O,O,0,13,16
3,3,platform,platform,NN,O,O,0,18,25
4,4,supports,support,VBZ,O,O,0,27,34
5,5,real-time,real-time,JJ,O,O,0,36,44
6,6,event,event,NN,O,O,0,46,50
7,7,streaming,stream,VBG,O,O,0,52,60
8,8,and,and,CC,O,O,0,62,64
9,9,identity,identity,NN,O,O,0,66,73


In [20]:
domain_rules = [
    {
        "id": "domain_term",
        "label": "DOMAIN_TERM",
        "priority": 20,
        "patterns": [
            [{"LOWER": "customer"}, {"LOWER": "data"}, {"LOWER": "platform"}],
            [{"LOWER": "real-time", "OP": "?"}, {"LOWER": "event"}, {"LEMMA": "stream"}],
            [{"LOWER": "identity"}, {"LOWER": "resolution"}],
            [{"LOWER": "enterprise"}, {"LEMMA": "account"}],
        ],
    }
]

domain_result, _ = run_rules(domain_rules)
display(match_table(domain_result))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,4,domain,customer data platform,domain_term,DOMAIN_TERM,0,4,25,1,3,1,3,20,0
1,4,domain,real-time event streaming,domain_term,DOMAIN_TERM,0,36,60,5,7,5,7,20,1
2,4,domain,event streaming,domain_term,DOMAIN_TERM,0,46,60,6,7,6,7,20,1
3,4,domain,identity resolution,domain_term,DOMAIN_TERM,0,66,84,9,10,9,10,20,2
4,4,domain,enterprise accounts,domain_term,DOMAIN_TERM,0,90,108,12,13,12,13,20,3


## **13. A Larger Rule Collection**

For real projects, keep related rules together and assign labels, IDs, and priorities intentionally. The collection below combines the address, contact, job, finance, and domain examples.

In [21]:
rule_collection = address_rules + contact_rules + job_rules + finance_rules + domain_rules

collection_result, collection_model = run_rules(
    rule_collection,
    output_col="all_rule_matches",
    overlap_strategy="PRIORITY_LONGEST",
)

all_matches = match_table(collection_result, "all_rule_matches")
display(all_matches.sort_values(["doc_id", "char_begin", "rule_id"]).reset_index(drop=True))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,0,address,"443 8th Street, New York",postal_address,ADDRESS,0,0,23,0,5,0,5,50,0
1,0,address,"21-B Baker Road, London",postal_address,ADDRESS,1,26,48,0,4,7,11,50,1
2,0,address,"1600 Pennsylvania Avenue, Washington",postal_address,ADDRESS,2,51,86,0,4,13,17,50,1
3,1,contact,"Jane Smith, Senior Data Scientist",person_with_title,PERSON_TITLE,0,0,32,0,5,0,5,30,0
4,1,contact,Example Corp in Berlin,organization_in_location,ORG_LOCATION,0,42,63,8,11,8,11,25,0
5,1,contact,phone: +1 415 555 0100,phone_label_value,PHONE,1,71,92,1,6,14,19,20,0
6,1,contact,email: jane.smith@example.com,email_label_value,EMAIL,1,97,125,8,10,21,23,20,0
7,2,jobs,3+ years of Python experience,years_of_skill_experience,EXPERIENCE_REQUIREMENT,0,8,36,2,6,2,6,40,0
8,2,jobs,Bachelor's degree in computer science,degree_requirement,EDUCATION_REQUIREMENT,2,124,160,0,4,24,28,30,0
9,3,finance,Revenue increased by 18 percent,revenue_change_first_sentence,REVENUE_CHANGE,0,0,30,0,4,0,4,45,0


## **14. Visualize Rule Matches With spark-nlp-display**

`RuleBasedMatcher` returns `CHUNK` annotations with `entity` metadata, so `spark-nlp-display` can render the final non-overlapping rule collection as highlighted spans. The table above is still the best place to inspect exact metadata; the visual layer is for quick span QA.

In [28]:
rule_label_colors = {
    "ADDRESS": "#4C78A8",
    "PERSON_TITLE": "#F58518",
    "ORG_LOCATION": "#54A24B",
    "PHONE": "#E45756",
    "EMAIL": "#B279A2",
    "EXPERIENCE_REQUIREMENT": "#72B7B2",
    "EDUCATION_REQUIREMENT": "#EECA3B",
    "REVENUE_CHANGE": "#59A14F",
    "LOSS_AMOUNT": "#E15759",
    "ACQUISITION": "#9C755F",
    "HEADQUARTERS": "#76B7B2",
    "DOMAIN_TERM": "#EDC948",
}

visual_docs = [
    (0, "Address rules"),
    (1, "Contact and identity rules"),
    (2, "Job requirement rules"),
    (3, "Finance and business rules"),
    (4, "Domain terminology rules"),
]

rule_vis = visualizer().set_label_colors(rule_label_colors)
for doc_id, title in visual_docs:
    row = (
        collection_result
        .where(F.col("doc_id") == doc_id)
        .select("document", "all_rule_matches")
        .first()
    )
    display(HTML(f"<h4 style='font-family: sans-serif; margin: 1rem 0 0.25rem'>{title}</h4>"))
    rule_vis.display(row.asDict(), label_col="all_rule_matches", document_col="document")

### Reading output metadata

`RuleBasedMatcher` emits `CHUNK` annotations. The metadata tells you which rule matched and where the match starts and ends in both sentence-local and document-level token indexes. The legacy `tokenBegin`/`tokenEnd` fields are sentence-local; prefer the explicit `sentenceToken*` and `documentToken*` fields in new code.

In [29]:
first_match = collection_result.select(F.explode("all_rule_matches").alias("match")).first().match
print("Matched text:", first_match.result)
print("Character span:", first_match.begin, first_match.end)
print(json.dumps(first_match.metadata, indent=2, sort_keys=True))

Matched text: 443 8th Street, New York
Character span: 0 23
{
  "chunk": "0",
  "document": "document:0:0:87:0",
  "documentKey": "document:0:0:87:0",
  "documentTokenBegin": "0",
  "documentTokenEnd": "5",
  "entity": "ADDRESS",
  "label": "ADDRESS",
  "pattern": "0",
  "priority": "50",
  "rule": "postal_address",
  "sentence": "0",
  "sentenceTokenBegin": "0",
  "sentenceTokenEnd": "5",
  "tokenBegin": "0",
  "tokenEnd": "5"
}


## **15. Loading Rules From an External JSONL Resource**

Python dictionaries/lists are convenient in notebooks. For production, many teams keep one JSON object per line in a versioned file. The matcher supports both styles. This cell writes JSONL to a temporary runtime directory and removes it when the notebook finishes.

In [30]:
tmp_rules_dir = tempfile.mkdtemp(prefix="rulebasedmatcher_rules_")
jsonl_path = Path(tmp_rules_dir) / "business_rules.jsonl"

with jsonl_path.open("w", encoding="utf-8") as f:
    for rule in finance_rules + domain_rules:
        f.write(json.dumps(rule) + "\n")

external_matcher = RuleBasedMatcher() \
    .setInputCols(["document", "sentence", "token", "lemma", "pos", "ner"]) \
    .setOutputCol("external_rule_matches") \
    .setRulesResource(str(jsonl_path), ReadAs.TEXT, {"format": "text"}) \
    .setAttributeColumns({
        "TEXT": "token",
        "LOWER": "token",
        "LEMMA": "lemma",
        "POS": "pos",
        "NER": "ner",
        "NER_TAG": "ner",
        "NER_TYPE": "ner",
    }) \
    .setOverlapStrategy("PRIORITY_LONGEST")

external_model = external_matcher.fit(annotated)
external_result = external_model.transform(annotated)

display(match_table(external_result, "external_rule_matches"))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,3,finance,Revenue increased by 18 percent,revenue_change_first_sentence,REVENUE_CHANGE,0,0,30,0,4,0,4,45,0
1,3,finance,Operating loss of $2.4 million,operating_loss_amount,LOSS_AMOUNT,1,33,62,0,4,6,10,40,0
2,3,finance,acquired by Example Corp,acquired_by_org,ACQUISITION,2,94,117,3,6,17,20,35,0
3,3,finance,headquartered in Berlin,headquartered_in_location,HEADQUARTERS,2,126,148,9,11,23,25,35,0
4,4,domain,customer data platform,domain_term,DOMAIN_TERM,0,4,25,1,3,1,3,20,0
5,4,domain,real-time event streaming,domain_term,DOMAIN_TERM,0,36,60,5,7,5,7,20,1
6,4,domain,identity resolution,domain_term,DOMAIN_TERM,0,66,84,9,10,9,10,20,2
7,4,domain,enterprise accounts,domain_term,DOMAIN_TERM,0,90,108,12,13,12,13,20,3


In [31]:
inline_subset, _ = run_rules(finance_rules + domain_rules, output_col="inline_subset", overlap_strategy="PRIORITY_LONGEST")
inline_rows = match_table(inline_subset, "inline_subset")[["doc_id", "matched_text", "rule_id", "label"]]
external_rows = match_table(external_result, "external_rule_matches")[["doc_id", "matched_text", "rule_id", "label"]]

print("Inline and external JSONL results are equivalent:", inline_rows.equals(external_rows))
shutil.rmtree(tmp_rules_dir, ignore_errors=True)

Inline and external JSONL results are equivalent: True


## **16. Complete Pipeline With RuleBasedMatcher**

The previous examples fit the matcher on an already annotated DataFrame for readability. In production you usually put the matcher at the end of a Spark ML pipeline. This example builds a lightweight full pipeline that can be saved and reloaded quickly.

In [32]:
light_rules = [
    {
        "id": "phone_value_light_pipeline",
        "label": "PHONE",
        "patterns": [[
            {"LOWER": "phone"},
            {"TEXT": ":", "OP": "?"},
            {"TEXT": {"REGEX": "^\\+?\\d+$"}, "OP": "{3,4}"},
        ]],
    },
    {
        "id": "email_value_light_pipeline",
        "label": "EMAIL",
        "patterns": [[
            {"LOWER": "email"},
            {"TEXT": ":", "OP": "?"},
            {"TEXT": {"REGEX": "^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$"}},
        ]],
    },
]

light_matcher = RuleBasedMatcher() \
    .setInputCols(["document", "sentence", "token"]) \
    .setOutputCol("contact_matches") \
    .setRules(light_rules) \
    .setAttributeColumns({"TEXT": "token", "LOWER": "token"})

light_pipeline = Pipeline(stages=[document, sentence, token, light_matcher])
light_model = light_pipeline.fit(df)
light_result = light_model.transform(df)

display(match_table(light_result, "contact_matches"))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,1,contact,phone: +1 415 555 0100,phone_value_light_pipeline,PHONE,1,71,92,1,6,14,19,0,0
1,1,contact,email: jane.smith@example.com,email_value_light_pipeline,EMAIL,1,97,125,8,10,21,23,0,0


## **17. Save and Reload the Pipeline**

Saved Spark ML pipelines preserve the parsed rule definitions and matcher parameters. The temporary directory is removed after the check.

In [33]:
tmp_model_dir = tempfile.mkdtemp(prefix="rulebasedmatcher_pipeline_")
try:
    light_model.write().overwrite().save(tmp_model_dir)
    reloaded = PipelineModel.load(tmp_model_dir)
    reloaded_result = reloaded.transform(df)
    display(match_table(reloaded_result, "contact_matches"))
finally:
    shutil.rmtree(tmp_model_dir, ignore_errors=True)

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,1,contact,phone: +1 415 555 0100,phone_value_light_pipeline,PHONE,1,71,92,1,6,14,19,0,0
1,1,contact,email: jane.smith@example.com,email_value_light_pipeline,EMAIL,1,97,125,8,10,21,23,0,0


## **18. Diagnostics: Invalid Rules and Alignment Problems**

The matcher validates rules and input mappings during `fit`. Catching these exceptions early is better than silently returning no matches.

In [34]:
def show_error(title, fn):
    print("\n" + title)
    try:
        fn()
    except Exception as exc:
        message = str(exc).split("\n")[0]
        print(message[:900])


show_error(
    "Unsupported operator",
    lambda: make_matcher([
        {"id": "bad_operator", "patterns": [[{"TEXT": {"ENDS_WITH": "ing"}}]]}
    ]).fit(annotated),
)

show_error(
    "Invalid quantifier",
    lambda: make_matcher([
        {"id": "bad_quantifier", "patterns": [[{"LOWER": "python", "OP": "{3,1}"}]]}
    ]).fit(annotated),
)

show_error(
    "Multiple TOKEN columns without base-token mapping",
    lambda: RuleBasedMatcher()
        .setInputCols(["document", "sentence", "token", "lemma"])
        .setOutputCol("bad_matches")
        .setRules([{"id": "lemma_only", "patterns": [[{"LEMMA": "experience"}]]}])
        .setAttributeColumns({"LEMMA": "lemma"})
        .fit(annotated),
)


Unsupported operator
Unsupported RuleBasedMatcher predicate 'ENDS_WITH' for attribute 'TEXT' in rule 'bad_operator' pattern 0 token 0

Invalid quantifier
requirement failed: Invalid RuleBasedMatcher quantifier '{3,1}' in rule 'bad_quantifier' pattern 0 token 0: max must be >= min

Multiple TOKEN columns without base-token mapping
requirement failed: RuleBasedMatcher found multiple TOKEN input columns (token, lemma). Map TEXT, TOKEN, or LOWER to the base token column with setAttributeColumns.


## **19. Choosing the Right Matcher**

Use `RuleBasedMatcher` when a rule needs token-aligned annotations such as POS, lemma, NER, or metadata.

Use other Spark NLP matchers when they fit the problem more directly:

* `RegexMatcher`: character-level regular expressions over documents.
* `TextMatcher` / `BigTextMatcher`: exact phrase dictionaries over tokens.
* `EntityRuler`: entity pattern resources, especially when you want entity-ruler behavior and storage options.
* `Chunker`: POS-sequence chunking with its existing regex grammar.

A good `RuleBasedMatcher` rule is selective early, bounded when using wildcards, and easy to explain from the token table.

## **20. Performance and Rule-Authoring Guidance**

Practical tips:

* Start from an annotation table, not from raw text guesses.
* Prefer exact, `IN`, POS, NER, or lemma predicates near the start of a pattern.
* Use bounded wildcards such as `{0,3}` instead of broad `*` when possible.
* Keep regexes specific and test them separately.
* Use `PRIORITY_LONGEST` for extraction workflows where one best chunk is preferred.
* Keep large rule collections in JSONL files and version them with your application.
* If a negated predicate such as `NOT_IN` does not match, check whether the attribute is missing. Negated predicates require the attribute to exist; use `EXISTS: false` when missing values are intentional.

## **21. Exercise**

Add a rule that extracts `event streaming` even when an optional adjective appears before it, such as `real-time event streaming` or `secure event streaming`.

Hint: combine an optional wildcard or POS predicate with `LOWER` and `LEMMA`.

In [35]:
exercise_rules = [
    {
        "id": "event_streaming_with_optional_modifier",
        "label": "DOMAIN_TERM",
        "patterns": [[
            {"POS": {"IN": ["JJ", "NN"]}, "OP": "?"},
            {"LOWER": "event"},
            {"LEMMA": "stream"},
        ]],
    }
]

exercise_result, _ = run_rules(exercise_rules)
display(match_table(exercise_result))

,doc_id,use_case,matched_text,rule_id,label,sentence,char_begin,char_end,sentence_token_begin,sentence_token_end,document_token_begin,document_token_end,priority,pattern
0,4,domain,real-time event streaming,event_streaming_with_optional_modifier,DOMAIN_TERM,0,36,60,5,7,5,7,0,0
1,4,domain,event streaming,event_streaming_with_optional_modifier,DOMAIN_TERM,0,46,60,6,7,6,7,0,0


## **Summary**

You have now used `RuleBasedMatcher` to:

* build a complete upstream Spark NLP annotation pipeline;
* inspect token, lemma, POS, NER, and metadata attributes before writing rules;
* define inline Python rule dictionaries and external JSONL rule resources;
* use equality, `IN`, `REGEX`, negation, `EXISTS`, wildcards, optional tokens, and repetitions;
* distinguish raw NER tags from normalized NER types;
* control overlapping matches with priorities;
* read match metadata and token indexes;
* save and reload a Spark ML pipeline containing the matcher.